# RT-DETR Phone Detection Training on Kaggle

This notebook fine-tunes `rtdetr-l.pt` for one-class phone detection.

Recommended Kaggle settings:
- Accelerator: GPU T4 x2
- Internet: ON for the first run, so `ultralytics` can be installed if needed
- Dataset: add `mobilephone_wusj2_v7_rtdetr_kaggle.zip` as a Kaggle Dataset

Main output:
- `phone_rtdetr_l_best.pt`
- `phone_rtdetr_l_last.pt`
- `phone_rtdetr_training_results.zip`


In [ ]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def ensure_package(package_name, import_name=None):
    import importlib.util
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f'Installing {package_name} without dependency upgrades...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', package_name])
    else:
        print(f'{package_name} is already available')

ensure_package('ultralytics')

import torch
import yaml
from ultralytics import RTDETR

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))


## 1. Locate and normalize the dataset

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
RUNS_DIR = WORKING_DIR / 'runs'
RUN_NAME = 'phone_rtdetr_l_t4_512'

yaml_candidates = sorted(INPUT_ROOT.rglob('data.yaml'))
if not yaml_candidates:
    zip_candidates = sorted(INPUT_ROOT.rglob('*.zip'))
    if zip_candidates:
        extract_root = WORKING_DIR / 'extracted_dataset'
        if extract_root.exists():
            shutil.rmtree(extract_root)
        extract_root.mkdir(parents=True, exist_ok=True)
        print('No extracted data.yaml found. Extracting:', zip_candidates[0])
        with zipfile.ZipFile(zip_candidates[0]) as zf:
            zf.extractall(extract_root)
        yaml_candidates = sorted(extract_root.rglob('data.yaml'))

if not yaml_candidates:
    raise FileNotFoundError('No data.yaml found under /kaggle/input or inside uploaded zip dataset.')

source_yaml = yaml_candidates[0]
dataset_root = source_yaml.parent
print('Using YAML:', source_yaml)
print('Dataset root:', dataset_root)

with open(source_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f) or {}

data_cfg['path'] = str(dataset_root)
data_cfg['train'] = 'train/images'
data_cfg['val'] = 'valid/images'
data_cfg['test'] = 'test/images'
data_cfg['nc'] = 1
data_cfg['names'] = ['Mobile-phone']

work_yaml = WORKING_DIR / 'mobilephone_rtdetr_data.yaml'
with open(work_yaml, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print(work_yaml.read_text())


## 2. Dataset sanity check

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def count_split(split):
    image_dir = dataset_root / split / 'images'
    label_dir = dataset_root / split / 'labels'
    images = [p for p in image_dir.glob('*') if p.suffix.lower() in IMG_EXTS]
    labels = list(label_dir.glob('*.txt')) if label_dir.exists() else []
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    boxes = 0
    bad_lines = 0
    class_ids = set()
    for label in labels:
        text = label.read_text(encoding='utf-8', errors='replace').strip()
        for line in text.splitlines():
            parts = line.split()
            if len(parts) != 5:
                bad_lines += 1
                continue
            try:
                cls = int(float(parts[0]))
                vals = list(map(float, parts[1:5]))
                class_ids.add(cls)
                if cls != 0 or any(v < 0 or v > 1 for v in vals) or vals[2] <= 0 or vals[3] <= 0:
                    bad_lines += 1
                boxes += 1
            except Exception:
                bad_lines += 1
    return {
        'images': len(images),
        'labels': len(labels),
        'missing_labels': len(image_stems - label_stems),
        'extra_labels': len(label_stems - image_stems),
        'boxes': boxes,
        'bad_label_lines': bad_lines,
        'class_ids': sorted(class_ids),
    }

stats = {split: count_split(split) for split in ['train', 'valid', 'test']}
stats


In [ ]:
for split, row in stats.items():
    assert row['images'] > 0, f'{split} has no images'
    assert row['labels'] > 0, f'{split} has no labels'
    assert row['missing_labels'] == 0, f'{split} has missing labels'
    assert row['bad_label_lines'] == 0, f'{split} has invalid label rows'
    assert row['class_ids'] == [0], f'{split} has unexpected classes: {row["class_ids"]}'
print('Dataset sanity check passed.')


## 3. Train RT-DETR-L

`rtdetr-l.pt` is the recommended starting point for this project because it is stronger than a nano detector while still designed for real-time detection. If T4x2 training fails, the code falls back to single GPU with a smaller batch.

In [ ]:
MODEL_NAME = 'rtdetr-l.pt'
EPOCHS = 60
IMGSZ = 512
BATCH_CANDIDATES = [4, 2, 1]
PATIENCE = 15

# RT-DETR-L is memory-heavy on T4. Single GPU is more stable than Kaggle DDP here.
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('Training device:', DEVICE)

def train_once(device, batch):
    model = RTDETR(MODEL_NAME)
    return model.train(
        data=str(work_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=batch,
        device=device,
        workers=2,
        patience=PATIENCE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        pretrained=True,
        optimizer='AdamW',
        lr0=0.0001,
        weight_decay=0.0005,
        cos_lr=True,
        amp=True,
        cache=False,
        deterministic=False,
        mosaic=0.0,
        scale=0.3,
        plots=True,
    )

last_error = None
for batch in BATCH_CANDIDATES:
    try:
        BATCH = batch
        print(f'Trying training with batch={BATCH}, imgsz={IMGSZ}, device={DEVICE}')
        train_results = train_once(DEVICE, BATCH)
        break
    except RuntimeError as exc:
        last_error = exc
        if 'out of memory' not in str(exc).lower():
            raise
        print(f'CUDA OOM with batch={batch}. Retrying with a smaller batch...')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    raise RuntimeError('RT-DETR-L training failed for all batch sizes.') from last_error


## 4. Validate on the test split

In [ ]:
run_dir = RUNS_DIR / 'detect' / RUN_NAME
best_pt = run_dir / 'weights' / 'best.pt'
last_pt = run_dir / 'weights' / 'last.pt'
assert best_pt.exists(), f'best.pt not found: {best_pt}'

best_model = RTDETR(str(best_pt))
metrics = best_model.val(
    data=str(work_yaml),
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME + '_test_eval',
    exist_ok=True,
    plots=True,
)
print(metrics)


## 5. Predict samples and package outputs

In [ ]:
sample_images = sorted((dataset_root / 'test' / 'images').glob('*'))[:24]
assert sample_images, 'No test images found for sample prediction'

best_model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(RUNS_DIR),
    name='rtdetr_predict_samples',
    exist_ok=True,
    save=True,
)

package_dir = WORKING_DIR / 'phone_rtdetr_export'
if package_dir.exists():
    shutil.rmtree(package_dir)
package_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(best_pt, package_dir / 'phone_rtdetr_l_best.pt')
if last_pt.exists():
    shutil.copy2(last_pt, package_dir / 'phone_rtdetr_l_last.pt')
shutil.copy2(work_yaml, package_dir / 'mobilephone_rtdetr_data.yaml')

for filename in ['results.csv', 'results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png']:
    src = run_dir / filename
    if src.exists():
        shutil.copy2(src, package_dir / filename)

zip_base = WORKING_DIR / 'phone_rtdetr_training_results'
zip_path = shutil.make_archive(str(zip_base), 'zip', package_dir)
print('Export folder:', package_dir)
print('Download zip:', zip_path)


## 6. Use the trained model in the local project

After downloading `phone_rtdetr_training_results.zip`, extract `phone_rtdetr_l_best.pt` and copy it to:

`D:\Work\HocKy3\ComputerVision\MDDAD\models\phone_rtdetr_l.pt`

Then test it on `/phone-test` by selecting the RT-DETR fine-tuned model or by entering the custom model path.